
# RHI Live Runtime v11 — Trace-Sufficient Recursive Agent

Δ **Purpose:** reset the runtime after the v9/v10 drift and build the next clean AI notebook.

This notebook keeps the core target narrow:

$$
\text{Prompt} \rightarrow \text{Need-Slot Contract} \rightarrow \text{Candidate Branches} \rightarrow
\text{Operational Audit} \rightarrow \text{Contract-Stance Gate} \rightarrow
\text{Recursive Repair} \rightarrow \Psi \text{ or } \Omega
$$

## What v11 fixes

1. **No missing `contract` argument.** The resolver receives the contract explicitly.
2. **No undefined `prompt`.** Every audit receives `prompt` as an argument.
3. **No nested parameter-sweep maze.** KRRB is implemented as an actual recursive resolver.
4. **No hard-coded paths.** The notebook treats the current folder as root.
5. **Shape-first, value-second.** It scores answers by operational fit, not surface vocabulary.
6. **Trace-sufficient output.** Every collapse saves enough evidence to inspect why it collapsed.

## Nexus interpretation

The prompt is not treated as a noun query. It is treated as an inverse cavity:

$$
Q \mapsto C_Q = (N, F, B, T, \Psi)
$$

where:

- $N$ = inverse need / missing slot
- $F$ = preserved function
- $B$ = boundary conditions
- $T$ = traps / forbidden neighboring carriers
- $\Psi$ = executable collapse target

A candidate answer is valid only when it occupies the cavity without merely repeating Nexus vocabulary.



In [15]:

# Optional install cell.
# Run this only if your environment is missing packages.
#
# %pip install -U pandas numpy torch transformers accelerate safetensors sentencepiece



In [16]:

from __future__ import annotations

import os
import re
import json
import math
import time
import uuid
import random
import traceback
from dataclasses import dataclass, asdict, field
from pathlib import Path
from collections import Counter, defaultdict
from typing import Any, Dict, List, Tuple, Optional

import numpy as np
import pandas as pd

ROOT = Path.cwd()
OUT_DIR = ROOT / "rhi_v11_outputs"
OUT_DIR.mkdir(exist_ok=True)

RUN_ID = "rhi_v11_" + uuid.uuid4().hex[:10]

print("ROOT:", ROOT)
print("OUT_DIR:", OUT_DIR)
print("RUN_ID:", RUN_ID)


ROOT: D:\Nexus\Nexus Mark 9\NoteBooks
OUT_DIR: D:\Nexus\Nexus Mark 9\NoteBooks\rhi_v11_outputs
RUN_ID: rhi_v11_c4aede871c



## Runtime configuration

Use your local model path if you have it downloaded in the same folder as this notebook, or leave the Hugging Face model ID.

For your RTX 4060, the safe default is a small instruct model:

$$
\text{Qwen2.5-1.5B-Instruct}
$$

The notebook will still run if the model cannot load. In that case, it uses deterministic branch generation so the controller logic can still be tested.


In [17]:

# Set this to a local folder if your model is already downloaded.
# Examples:
# MODEL_ID_OR_PATH = "./Qwen2.5-1.5B-Instruct"
# MODEL_ID_OR_PATH = "C:/Users/Dean/Downloads/Qwen2.5-1.5B-Instruct"
# MODEL_ID_OR_PATH = "Qwen/Qwen2.5-1.5B-Instruct"

MODEL_ID_OR_PATH = os.environ.get("RHI_MODEL", "Qwen/Qwen2.5-1.5B-Instruct")

LOAD_REAL_MODEL = True
MAX_NEW_TOKENS = 360
TEMPERATURE = 0.45
TOP_P = 0.90
SEED = 7

random.seed(SEED)
np.random.seed(SEED)

print("MODEL_ID_OR_PATH:", MODEL_ID_OR_PATH)
print("LOAD_REAL_MODEL:", LOAD_REAL_MODEL)


MODEL_ID_OR_PATH: Qwen/Qwen2.5-1.5B-Instruct
LOAD_REAL_MODEL: True


In [18]:

# Optional local model loader.
# This cell catches failures and keeps the notebook running.

tokenizer = None
model = None
MODEL_READY = False
DEVICE_INFO = {}

def try_load_model(model_id_or_path: str):
    global tokenizer, model, MODEL_READY, DEVICE_INFO

    if not LOAD_REAL_MODEL:
        print("Model loading disabled. Using deterministic fallback branches.")
        return False

    try:
        import torch
        from transformers import AutoTokenizer, AutoModelForCausalLM

        DEVICE_INFO["torch_version"] = torch.__version__
        DEVICE_INFO["cuda_available"] = bool(torch.cuda.is_available())
        DEVICE_INFO["device_count"] = int(torch.cuda.device_count())
        if torch.cuda.is_available():
            DEVICE_INFO["gpu_name"] = torch.cuda.get_device_name(0)
            DEVICE_INFO["cuda_version"] = torch.version.cuda

        print("Torch/CUDA:", DEVICE_INFO)

        tokenizer = AutoTokenizer.from_pretrained(model_id_or_path, trust_remote_code=True)

        dtype = torch.float16 if torch.cuda.is_available() else torch.float32
        model = AutoModelForCausalLM.from_pretrained(
            model_id_or_path,
            trust_remote_code=True,
            torch_dtype=dtype,
            device_map="auto" if torch.cuda.is_available() else None,
            low_cpu_mem_usage=True,
        )

        model.eval()
        MODEL_READY = True
        print("MODEL_READY:", MODEL_READY)
        return True

    except Exception as e:
        MODEL_READY = False
        print("MODEL LOAD FAILED — using deterministic fallback branches.")
        print(type(e).__name__ + ":", e)
        return False

_ = try_load_model(MODEL_ID_OR_PATH)


Torch/CUDA: {'torch_version': '2.11.0+cu126', 'cuda_available': True, 'device_count': 1, 'gpu_name': 'NVIDIA GeForce RTX 4060', 'cuda_version': '12.6'}


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

MODEL_READY: True



## Core vocabulary

The controller needs framework language, but it must not be fooled by framework language.  
So v11 separates:

$$
\text{surface overlap} \neq \text{operational agreement}
$$

That is the main correction after v4/v5.


In [19]:

STOPWORDS = {
    "the","a","an","and","or","but","if","then","else","of","to","in","on","for","with","by","as",
    "is","are","was","were","be","being","been","it","this","that","these","those","from","at",
    "into","out","about","so","because","therefore","than","not","no","yes","do","does","did",
    "can","could","should","would","will","just","they","them","their","you","your","we","our",
    "i","me","my","he","she","his","her","its"
}

# Nexus terms get their own stop-band for lexical consensus.
# This prevents false Ψ just because two answers both say "contract / collapse / carrier".
NEXUS_SURFACE_TERMS = {
    "nexus","contract","carrier","domain","boundary","collapse","shape","value","slot","need",
    "forbidden","neighbor","operational","recursive","recursion","krrb","omega","psi","field",
    "fold","runtime","phase","lock","audit","trace","signal","evidence","branch","repair"
}

def words(text: str, remove_nexus_surface: bool = False) -> List[str]:
    toks = re.findall(r"[a-zA-Z0-9_ΔΨΩ⊕↻⊥]+", str(text).lower())
    toks = [t for t in toks if t not in STOPWORDS and len(t) > 1]
    if remove_nexus_surface:
        toks = [t for t in toks if t not in NEXUS_SURFACE_TERMS]
    return toks

def wordset(text: str, remove_nexus_surface: bool = False) -> set:
    return set(words(text, remove_nexus_surface=remove_nexus_surface))

def jaccard(a: str, b: str, remove_nexus_surface: bool = False) -> float:
    wa, wb = wordset(a, remove_nexus_surface), wordset(b, remove_nexus_surface)
    if not wa and not wb:
        return 1.0
    if not wa or not wb:
        return 0.0
    return len(wa & wb) / max(1, len(wa | wb))

def contains_any(text: str, terms: List[str]) -> bool:
    s = str(text).lower()
    return any(str(t).lower() in s for t in terms)

def clamp(x: float, lo: float = 0.0, hi: float = 1.0) -> float:
    return max(lo, min(hi, float(x)))

def harmonic_mean(vals: List[float], eps: float = 1e-9) -> float:
    vals = [max(eps, float(v)) for v in vals]
    return len(vals) / sum(1.0 / v for v in vals)

def normalized_entropy(weights: Dict[str, float]) -> float:
    xs = np.array([max(0.0, float(v)) for v in weights.values()], dtype=float)
    total = xs.sum()
    if total <= 0:
        return 1.0
    p = xs / total
    h = -float(np.sum([q * math.log(q + 1e-12) for q in p]))
    return h / math.log(len(xs)) if len(xs) > 1 else 0.0



## Shape templates

This is not a classifier in the usual ML sense. It is a small operational lens.

A prompt can invoke multiple templates. The answer must satisfy the active templates, not just sound like it belongs.


In [20]:

SHAPE_TEMPLATES = {
    "CONTRACT": {
        "triggers": ["contract", "before", "intent", "tool", "agent", "plan", "spec", "interface"],
        "needs": ["contract", "intent", "boundary", "tool", "before", "select", "gate"]
    },
    "GROOVE": {
        "triggers": ["train", "lora", "qlora", "adapter", "fine tune", "weights", "groove", "model"],
        "needs": ["adapter", "low-rank", "weights", "delta", "dataset", "loss", "eval"]
    },
    "SEARCH": {
        "triggers": ["search", "retrieve", "find", "query", "lookup", "index", "rag"],
        "needs": ["query", "retrieve", "candidate", "rank", "verify", "evidence"]
    },
    "REPAIR": {
        "triggers": ["fix", "repair", "error", "failed", "broken", "bug", "traceback", "syntaxerror", "nameerror"],
        "needs": ["failure", "cause", "patch", "test", "rerun", "trace"]
    },
    "MEMORY": {
        "triggers": ["remember", "memory", "recall", "lost", "state", "context"],
        "needs": ["state", "trace", "retrieve", "preserve", "update", "continuity"]
    },
    "BOUNDARY": {
        "triggers": ["boundary", "limit", "forbidden", "constraint", "safety", "gate", "reject"],
        "needs": ["boundary", "reject", "constraint", "preserve", "violate", "gate"]
    },
    "RECURSE": {
        "triggers": ["recursive", "recursion", "again", "loop", "fold", "iterate", "turn"],
        "needs": ["recursive", "branch", "feedback", "repair", "collapse", "omega"]
    },
    "TOOL": {
        "triggers": ["tool", "api", "call", "function", "agent", "execute", "act"],
        "needs": ["tool", "contract", "input", "output", "side-effect", "verify"]
    },
}

def detect_shape_template(prompt: str) -> List[str]:
    p = str(prompt).lower()
    active = []
    for name, cfg in SHAPE_TEMPLATES.items():
        hits = sum(1 for t in cfg["triggers"] if t in p)
        if hits:
            active.append(name)
    return active or ["GENERAL"]

def shape_mass(text: str, active_templates: List[str]) -> Dict[str, float]:
    masses = {}
    for name in active_templates:
        if name == "GENERAL":
            continue
        needs = SHAPE_TEMPLATES[name]["needs"]
        masses[name] = sum(1 for n in needs if n.lower() in str(text).lower()) / max(1, len(needs))
    if not masses:
        masses["GENERAL"] = 0.5
    return masses

def shape_score(text: str, active_templates: List[str]) -> float:
    masses = shape_mass(text, active_templates)
    return clamp(sum(masses.values()) / max(1, len(masses)))



## Need-slot contract

This is the v11 $\Delta$-point.

The answer does not start from available nouns. It starts from the missing executable shape.

$$
C_Q =
\left[
N_Q,\;F_Q,\;B_Q,\;T_Q,\;\Psi_Q
\right]
$$


In [21]:

@dataclass
class NeedSlotContract:
    prompt: str
    active_templates: List[str]
    inverse_need: str
    preserved_function: str
    boundary_conditions: List[str]
    domain_carrier: List[str]
    forbidden_neighbors: List[str]
    collapse_target: str
    repair_history: List[Dict[str, Any]] = field(default_factory=list)

def extract_domain_terms(prompt: str, max_terms: int = 14) -> List[str]:
    ws = words(prompt, remove_nexus_surface=True)
    counts = Counter(ws)
    return [w for w, _ in counts.most_common(max_terms)]

def infer_forbidden_neighbors(prompt: str, active: List[str]) -> List[str]:
    forb = set()

    if "TOOL" in active or "CONTRACT" in active:
        forb.update(["tool-first action", "premature execution", "api reflex", "surface task completion"])
    if "GROOVE" in active:
        forb.update(["full retrain reflex", "weight-churn", "dataset worship", "loss-only tuning"])
    if "SEARCH" in active:
        forb.update(["noun lookup", "keyword matching", "unverified retrieval", "search without verifier"])
    if "REPAIR" in active:
        forb.update(["blanket rewrite", "threshold fiddling", "silent failure", "patch without test"])
    if "MEMORY" in active:
        forb.update(["stateless answer", "context amnesia", "surface recall", "summary as memory"])
    if "BOUNDARY" in active:
        forb.update(["unsafe override", "constraint erasure", "boundary confusion"])
    if "RECURSE" in active:
        forb.update(["linear pipeline", "single branch", "dead loop", "nested sweep masquerading as recursion"])

    if not forb:
        forb.update(["surface label", "generic explanation", "noun-only answer"])

    return sorted(forb)

def build_contract(prompt: str) -> NeedSlotContract:
    active = detect_shape_template(prompt)
    domain_terms = extract_domain_terms(prompt)

    inverse_need = (
        "construct the missing operational slot implied by the prompt; "
        "select or generate only answers that preserve the required operation"
    )

    preserved_function_parts = []
    if "TOOL" in active or "CONTRACT" in active:
        preserved_function_parts.append("form contract before tool use")
    if "GROOVE" in active:
        preserved_function_parts.append("shape model behavior through low-rank/grooved update, not blind full retrain")
    if "SEARCH" in active:
        preserved_function_parts.append("retrieve candidates then verify by shape fit")
    if "REPAIR" in active:
        preserved_function_parts.append("repair the failed dimension and rerun")
    if "MEMORY" in active:
        preserved_function_parts.append("preserve trace/state across turns")
    if "RECURSE" in active:
        preserved_function_parts.append("branch recursively until Ψ collapse or Ω residue")
    if not preserved_function_parts:
        preserved_function_parts.append("preserve the prompt's verb-level operation")

    boundary_conditions = [
        "do not collapse on shared framework vocabulary alone",
        "require evidence trace for the selected answer",
        "prefer Ω over false Ψ when top branches disagree operationally",
        "preserve base answer when controller evidence is weak",
    ]

    collapse_target = (
        "one executable answer with contract, evidence, operational fit, and trace sufficient to debug"
    )

    return NeedSlotContract(
        prompt=prompt,
        active_templates=active,
        inverse_need=inverse_need,
        preserved_function="; ".join(preserved_function_parts),
        boundary_conditions=boundary_conditions,
        domain_carrier=domain_terms,
        forbidden_neighbors=infer_forbidden_neighbors(prompt, active),
        collapse_target=collapse_target,
    )

def contract_to_text(c: NeedSlotContract) -> str:
    return (
        f"ACTIVE_TEMPLATES: {', '.join(c.active_templates)}\n"
        f"INVERSE_NEED: {c.inverse_need}\n"
        f"PRESERVED_FUNCTION: {c.preserved_function}\n"
        f"BOUNDARY_CONDITIONS: {' | '.join(c.boundary_conditions)}\n"
        f"DOMAIN_CARRIER: {', '.join(c.domain_carrier)}\n"
        f"FORBIDDEN_NEIGHBORS: {' | '.join(c.forbidden_neighbors)}\n"
        f"COLLAPSE_TARGET: {c.collapse_target}"
    )



## Candidate branches

KRRB needs real branch diversity.  
Each branch gets the same contract but a different stance.

This is the simplest working triad:

$$
\text{Construct} \oplus \text{Verify} \oplus \text{Repair}
$$


In [22]:

BRANCH_SYSTEMS = {
    "construct": (
        "You are the constructor branch. Build the answer from the missing operational slot first. "
        "Do not start with labels. State the contract, the preserved function, and the executable answer."
    ),
    "verify": (
        "You are the verifier branch. Test the candidate answer against need, function, boundary, trap, and collapse. "
        "Prefer clear rejection over false agreement."
    ),
    "repair": (
        "You are the repair branch. Identify the likely failure mode, then produce the minimal repaired answer. "
        "Do not blanket-rewrite. Patch only the failed operational dimension."
    ),
    "counter": (
        "You are the counter-branch. Name the strongest wrong path and explain why it fails. "
        "Then give the corrected path."
    ),
}

def deterministic_branch(prompt: str, contract: NeedSlotContract, branch_name: str) -> str:
    # This fallback keeps the controller testable even without a loaded LLM.
    if branch_name == "construct":
        return (
            "Δ Contract-first answer: the prompt is asking for an AI runtime that forms a need-slot before acting. "
            f"The preserved function is: {contract.preserved_function}. "
            "The correct flow is prompt → contract → branch candidates → operational audit → gated collapse. "
            "The answer should not be a noun lookup; it should construct the inverse shape of the missing operation and then select the candidate that fits."
        )
    if branch_name == "verify":
        return (
            "Verification: current agents fail when they call tools before forming a contract because tool output becomes the driver instead of evidence. "
            "A contract defines intent, boundary conditions, forbidden neighbors, and collapse criteria. "
            "The verifier rejects branches that share vocabulary but disagree on the operation being preserved."
        )
    if branch_name == "repair":
        return (
            "Repair: if the runtime falls to Ω, do not widen thresholds. Extract the failed observable: need, function, boundary, trap, or collapse. "
            "Repair only that dimension, regenerate the candidate, and recurse. "
            "The repaired path keeps the base model protected unless support, audit, and stance agreement align."
        )
    if branch_name == "counter":
        return (
            "Wrong path: tool-first execution looks efficient, but it confuses action with resolution. "
            "The tool can return data while the agent still lacks the shape of what would count as success. "
            "Correct path: build the contract first, then use tools as constrained evidence channels."
        )
    return "No branch."

def model_generate_one(prompt: str, contract: NeedSlotContract, branch_name: str) -> str:
    if not MODEL_READY:
        return deterministic_branch(prompt, contract, branch_name)

    import torch

    system = BRANCH_SYSTEMS[branch_name]
    user = (
        "PROMPT:\n" + prompt.strip() + "\n\n"
        "NEED-SLOT CONTRACT:\n" + contract_to_text(contract) + "\n\n"
        "Return a compact but complete answer. Include operational evidence. Avoid generic wording."
    )

    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user},
    ]

    try:
        if hasattr(tokenizer, "apply_chat_template"):
            input_ids = tokenizer.apply_chat_template(
                messages,
                add_generation_prompt=True,
                return_tensors="pt"
            )
            input_ids = input_ids.to(model.device)
            with torch.no_grad():
                out = model.generate(
                    input_ids,
                    max_new_tokens=MAX_NEW_TOKENS,
                    do_sample=True,
                    temperature=TEMPERATURE,
                    top_p=TOP_P,
                    pad_token_id=tokenizer.eos_token_id,
                )
            gen = out[0][input_ids.shape[-1]:]
            return tokenizer.decode(gen, skip_special_tokens=True).strip()

        # fallback if tokenizer has no chat template
        text = system + "\n\n" + user + "\n\nANSWER:\n"
        inputs = tokenizer(text, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=True,
                temperature=TEMPERATURE,
                top_p=TOP_P,
                pad_token_id=tokenizer.eos_token_id,
            )
        return tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()

    except Exception as e:
        print(f"Model generation failed for branch={branch_name}. Using deterministic fallback.", type(e).__name__, e)
        return deterministic_branch(prompt, contract, branch_name)

def generate_candidates(prompt: str, contract: NeedSlotContract) -> List[Dict[str, Any]]:
    candidates = []
    for branch_name in BRANCH_SYSTEMS:
        txt = model_generate_one(prompt, contract, branch_name)
        candidates.append({
            "branch": branch_name,
            "answer": txt,
        })
    return candidates



## Operational audit

The five-dimensional audit is the spine:

$$
A_i =
(F_{\text{need}}, F_{\text{function}}, F_{\text{boundary}}, F_{\text{trap}}, F_{\text{collapse}})
$$

The old failure was checking what the answer **said**.  
The new check asks what the answer **does**.


In [23]:

def field_hit_score(text: str, field_terms: List[str]) -> float:
    if not field_terms:
        return 0.5
    s = str(text).lower()
    hits = 0
    for term in field_terms:
        term = str(term).lower()
        if not term:
            continue
        if term in s:
            hits += 1
    return clamp(hits / max(1, min(len(field_terms), 8)))

def contract_field_terms(contract: NeedSlotContract) -> Dict[str, List[str]]:
    return {
        "need": words(contract.inverse_need, remove_nexus_surface=True),
        "function": words(contract.preserved_function, remove_nexus_surface=True),
        "boundary": words(" ".join(contract.boundary_conditions), remove_nexus_surface=True),
        "domain": contract.domain_carrier,
        "forbidden": words(" ".join(contract.forbidden_neighbors), remove_nexus_surface=True),
        "collapse": words(contract.collapse_target, remove_nexus_surface=True),
    }

def answer_operational_audit(prompt: str, contract: NeedSlotContract, answer: str) -> Dict[str, Any]:
    # prompt is explicit here — this fixes the v10 NameError class.
    active_templates = detect_shape_template(prompt)
    fields = contract_field_terms(contract)
    a = str(answer).lower()

    F_need = clamp(0.55 * field_hit_score(answer, fields["need"]) + 0.45 * field_hit_score(answer, fields["domain"]))

    F_function = clamp(
        0.60 * field_hit_score(answer, fields["function"]) +
        0.40 * sum([
            contains_any(a, ["preserve", "continues", "maintain", "function", "operation", "before", "after"]),
            contains_any(a, ["flow", "runtime", "execute", "candidate", "select", "verify"]),
        ]) / 2
    )

    F_boundary = clamp(
        0.60 * field_hit_score(answer, fields["boundary"]) +
        0.40 * sum([
            contains_any(a, ["boundary", "constraint", "gate", "reject", "protect", "forbidden"]),
            contains_any(a, ["false", "wrong", "weak", "unsafe", "premature", "surface"]),
        ]) / 2
    )

    # Trap score is high when answer explicitly rejects forbidden neighbors.
    forbidden_hit = field_hit_score(answer, fields["forbidden"])
    trap_language = sum([
        contains_any(a, ["not", "instead", "wrong", "fails", "reject", "avoid", "forbidden"]),
        contains_any(a, ["tool-first", "surface", "generic", "threshold", "noun", "keyword"]),
    ]) / 2
    F_trap = clamp(0.45 * forbidden_hit + 0.55 * trap_language)

    F_collapse = clamp(
        0.50 * field_hit_score(answer, fields["collapse"]) +
        0.50 * sum([
            contains_any(a, ["therefore", "so", "because", "result", "collapse", "answer"]),
            contains_any(a, ["one", "single", "executable", "run", "test", "trace"]),
        ]) / 2
    )

    masses = shape_mass(answer, active_templates)
    F_shape = shape_score(answer, active_templates)

    # Template-aware boost, but prompt is passed in directly.
    template_requirements = {}
    for template in active_templates:
        if template == "GENERAL":
            continue
        needs = SHAPE_TEMPLATES[template]["needs"]
        template_requirements[template] = field_hit_score(answer, needs)

    # Hot/cold: hot = operation pressure; cold = constraint/boundary preservation.
    hot = clamp((F_need + F_function + F_shape) / 3)
    cold = clamp((F_boundary + F_trap + F_collapse) / 3)
    hotcold_balance = clamp(1.0 - abs(hot - cold))

    quality = harmonic_mean([F_need, F_function, F_boundary, F_trap, F_collapse])
    mean_quality = float(np.mean([F_need, F_function, F_boundary, F_trap, F_collapse]))

    return {
        "F_need": F_need,
        "F_function": F_function,
        "F_boundary": F_boundary,
        "F_trap": F_trap,
        "F_collapse": F_collapse,
        "F_shape": F_shape,
        "hot": hot,
        "cold": cold,
        "hotcold_balance": hotcold_balance,
        "quality_hmean": quality,
        "quality_mean": mean_quality,
        "shape_mass": masses,
        "template_requirements": template_requirements,
        "active_templates": active_templates,
    }

def audit_agreement(audit_a: Dict[str, Any], audit_b: Dict[str, Any]) -> float:
    keys = ["F_need", "F_function", "F_boundary", "F_trap", "F_collapse"]
    diffs = [abs(float(audit_a[k]) - float(audit_b[k])) for k in keys]
    return clamp(1.0 - float(np.mean(diffs)))

def contract_stance_agreement(contract: NeedSlotContract, answer_a: str, answer_b: str) -> Dict[str, Any]:
    # Contract-grounded agreement: do both branches stake the same operational fields?
    fields = contract_field_terms(contract)
    scores = {}
    for field_name in ["need", "function", "boundary", "domain", "collapse"]:
        terms = fields[field_name]
        sa = set(t for t in terms if str(t).lower() in str(answer_a).lower())
        sb = set(t for t in terms if str(t).lower() in str(answer_b).lower())
        if not terms:
            score = 0.5
        elif not sa and not sb:
            score = 0.25
        else:
            score = len(sa & sb) / max(1, len(sa | sb))
        scores[field_name] = clamp(score)

    # Forbidden stance: if both reject the same traps, that is agreement.
    fterms = fields["forbidden"]
    fa = set(t for t in fterms if str(t).lower() in str(answer_a).lower())
    fb = set(t for t in fterms if str(t).lower() in str(answer_b).lower())
    if fterms:
        scores["forbidden"] = clamp(len(fa & fb) / max(1, len(fa | fb))) if (fa or fb) else 0.25
    else:
        scores["forbidden"] = 0.5

    aggregate = harmonic_mean(list(scores.values()))
    return {
        "aggregate": aggregate,
        "fields": scores,
    }



## Score and trace

A branch is not “good” because it sounds fluent.  
It is good when its independent channels phase-lock.

$$
S_i =
0.32 A_i +
0.22 K_i +
0.18 M_i +
0.16 T_i +
0.12 R_i
$$

where:

- $A_i$ = operational audit
- $K_i$ = contract field stance
- $M_i$ = shape mass
- $T_i$ = trace sufficiency
- $R_i$ = hot/cold balance


In [24]:

def trace_sufficiency(answer: str, audit: Dict[str, Any], contract: NeedSlotContract) -> float:
    a = str(answer).lower()
    evidence_bits = [
        contains_any(a, ["because", "therefore", "so", "why"]),
        contains_any(a, ["contract", "intent", "boundary", "collapse"]),
        contains_any(a, ["tool", "agent", "runtime", "model", "candidate"]),
        contains_any(a, ["verify", "evidence", "trace", "audit", "test"]),
        audit["quality_hmean"] >= 0.45,
    ]
    return clamp(sum(evidence_bits) / len(evidence_bits))

def branch_score(prompt: str, contract: NeedSlotContract, branch: Dict[str, Any]) -> Dict[str, Any]:
    answer = branch["answer"]
    audit = answer_operational_audit(prompt, contract, answer)
    fields = contract_field_terms(contract)

    contract_stance = np.mean([
        field_hit_score(answer, fields["need"]),
        field_hit_score(answer, fields["function"]),
        field_hit_score(answer, fields["boundary"]),
        field_hit_score(answer, fields["domain"]),
        field_hit_score(answer, fields["collapse"]),
    ])

    trace = trace_sufficiency(answer, audit, contract)
    shape = audit["F_shape"]
    hotcold = audit["hotcold_balance"]

    score = clamp(
        0.32 * audit["quality_hmean"] +
        0.22 * contract_stance +
        0.18 * shape +
        0.16 * trace +
        0.12 * hotcold
    )

    return {
        **branch,
        "score": score,
        "contract_stance": float(contract_stance),
        "trace_sufficiency": trace,
        "audit": audit,
    }

def score_candidates(prompt: str, contract: NeedSlotContract, candidates: List[Dict[str, Any]]) -> pd.DataFrame:
    rows = [branch_score(prompt, contract, c) for c in candidates]
    flat = []
    for r in rows:
        audit = r["audit"]
        flat.append({
            "branch": r["branch"],
            "score": r["score"],
            "contract_stance": r["contract_stance"],
            "trace_sufficiency": r["trace_sufficiency"],
            "quality_hmean": audit["quality_hmean"],
            "F_need": audit["F_need"],
            "F_function": audit["F_function"],
            "F_boundary": audit["F_boundary"],
            "F_trap": audit["F_trap"],
            "F_collapse": audit["F_collapse"],
            "F_shape": audit["F_shape"],
            "hot": audit["hot"],
            "cold": audit["cold"],
            "hotcold_balance": audit["hotcold_balance"],
            "answer": r["answer"],
            "detail": r,
        })
    return pd.DataFrame(flat).sort_values("score", ascending=False).reset_index(drop=True)



## Recursive KRRB resolver

This is the part that replaces the nested sweep.

A bad branch does not trigger parameter search.  
It triggers a **structured Ω**, then a **targeted contract repair**, then recursion.

$$
C_{t+1} = \operatorname{Repair}(C_t, \Omega_t)
$$

$$
\operatorname{KRRB}(Q,C_t,B_t) =
\begin{cases}
\Psi, & \text{if collapse gates pass}\\
\operatorname{KRRB}(Q,C_{t+1},B_{t+1}), & \text{if repair budget remains}\\
\Omega, & \text{otherwise}
\end{cases}
$$


In [25]:

PSI_MIN = 0.58
MARGIN_MIN = 0.045
CONSENSUS_MARGIN_MAX = 0.06
AUDIT_AGREE_MIN = 0.84
LEXICAL_AGREE_MIN = 0.28
STANCE_AGREE_MIN = 0.42
MAX_RECURSION_DEPTH = 3

def extract_omega(score_df: pd.DataFrame, contract: NeedSlotContract) -> Dict[str, Any]:
    top = score_df.iloc[0]
    audit = top["detail"]["audit"]
    dims = ["F_need", "F_function", "F_boundary", "F_trap", "F_collapse", "F_shape"]
    weak = {k: float(audit[k]) for k in dims if float(audit[k]) < 0.48}

    if not weak:
        weak = {"margin_or_consensus": 0.0}

    return {
        "weak_dimensions": weak,
        "top_branch": top["branch"],
        "top_score": float(top["score"]),
        "reason": "structured_operational_failure",
    }

def repair_contract(contract: NeedSlotContract, omega: Dict[str, Any]) -> NeedSlotContract:
    c = NeedSlotContract(**asdict(contract))
    weak = omega.get("weak_dimensions", {})
    repair_note = {"omega": omega, "time": time.time()}

    if "F_need" in weak:
        c.inverse_need += "; sharpen the inverse cavity and name what must be occupied"
    if "F_function" in weak:
        c.preserved_function += "; explicitly state the verb/function that must continue"
    if "F_boundary" in weak:
        c.boundary_conditions.append("explicitly name what the answer must not violate")
    if "F_trap" in weak:
        c.forbidden_neighbors.append("shared Nexus vocabulary without operational stance")
    if "F_collapse" in weak:
        c.collapse_target += "; return one runnable decision and its reason"
    if "F_shape" in weak:
        c.boundary_conditions.append("answer must satisfy active shape templates, not surface labels")

    c.repair_history.append(repair_note)
    return c

def repair_candidates(prompt: str, contract: NeedSlotContract, prev_score_df: pd.DataFrame, omega: Dict[str, Any]) -> List[Dict[str, Any]]:
    top_answer = str(prev_score_df.iloc[0]["answer"])
    weak = ", ".join(omega.get("weak_dimensions", {}).keys())

    repair_prompt = (
        prompt.strip()
        + "\n\nPrevious top candidate:\n"
        + top_answer
        + "\n\nΩ failure dimensions: "
        + weak
        + "\n\nRepair only the failed dimensions. Preserve correct structure. Return a stronger executable answer."
    )

    # Use the same branch machinery, but against repaired contract and repair prompt.
    return generate_candidates(repair_prompt, contract)

def consensus_gate(contract: NeedSlotContract, score_df: pd.DataFrame) -> Dict[str, Any]:
    if len(score_df) < 2:
        return {"ok": False, "reason": "not_enough_branches"}

    a = score_df.iloc[0]["detail"]
    b = score_df.iloc[1]["detail"]

    margin = float(score_df.iloc[0]["score"] - score_df.iloc[1]["score"])
    both_high = float(score_df.iloc[0]["score"]) >= PSI_MIN and float(score_df.iloc[1]["score"]) >= PSI_MIN - 0.03

    lex_plain = jaccard(a["answer"], b["answer"], remove_nexus_surface=False)
    lex_oper = jaccard(a["answer"], b["answer"], remove_nexus_surface=True)
    audit_ag = audit_agreement(a["audit"], b["audit"])
    stance = contract_stance_agreement(contract, a["answer"], b["answer"])

    # v11: surface vocabulary alone cannot collapse.
    lexical_ok = lex_oper >= LEXICAL_AGREE_MIN
    audit_ok = audit_ag >= AUDIT_AGREE_MIN
    stance_ok = stance["aggregate"] >= STANCE_AGREE_MIN

    ok = (
        both_high
        and margin <= CONSENSUS_MARGIN_MAX
        and stance_ok
        and (audit_ok or lexical_ok)
    )

    return {
        "ok": bool(ok),
        "reason": "contract_anchored_consensus" if ok else "no_consensus",
        "margin": margin,
        "both_high": bool(both_high),
        "lex_plain": float(lex_plain),
        "lex_operational": float(lex_oper),
        "audit_agreement": float(audit_ag),
        "stance_agreement": float(stance["aggregate"]),
        "stance_fields": stance["fields"],
    }

def direct_collapse_gate(score_df: pd.DataFrame) -> Dict[str, Any]:
    top = score_df.iloc[0]
    margin = float(top["score"] - score_df.iloc[1]["score"]) if len(score_df) > 1 else float(top["score"])
    ok = float(top["score"]) >= PSI_MIN and margin >= MARGIN_MIN and float(top["trace_sufficiency"]) >= 0.45
    return {
        "ok": bool(ok),
        "reason": "direct_margin_collapse" if ok else "no_direct_collapse",
        "margin": margin,
        "top_score": float(top["score"]),
        "trace_sufficiency": float(top["trace_sufficiency"]),
    }

def krrb_recursive_resolve(
    prompt: str,
    contract: NeedSlotContract,
    candidates: List[Dict[str, Any]],
    depth: int = 0,
    trace: Optional[List[Dict[str, Any]]] = None,
) -> Dict[str, Any]:

    if trace is None:
        trace = []

    score_df = score_candidates(prompt, contract, candidates)
    direct = direct_collapse_gate(score_df)
    consensus = consensus_gate(contract, score_df)

    step = {
        "depth": depth,
        "contract": asdict(contract),
        "scores": score_df.drop(columns=["detail"]).to_dict(orient="records"),
        "direct_gate": direct,
        "consensus_gate": consensus,
    }
    trace.append(step)

    if direct["ok"]:
        return {
            "state": "Ψ",
            "reason": direct["reason"],
            "winner": score_df.iloc[0]["detail"],
            "score_df": score_df,
            "trace": trace,
            "depth": depth,
        }

    if consensus["ok"]:
        # If consensus passes, use the higher scoring branch but record consensus proof.
        return {
            "state": "Ψ",
            "reason": consensus["reason"],
            "winner": score_df.iloc[0]["detail"],
            "score_df": score_df,
            "trace": trace,
            "depth": depth,
        }

    omega = extract_omega(score_df, contract)
    step["omega"] = omega

    if depth >= MAX_RECURSION_DEPTH:
        return {
            "state": "Ω",
            "reason": "max_recursion_depth",
            "omega": omega,
            "winner": score_df.iloc[0]["detail"],
            "score_df": score_df,
            "trace": trace,
            "depth": depth,
        }

    repaired_contract = repair_contract(contract, omega)
    repaired_candidates = repair_candidates(prompt, repaired_contract, score_df, omega)

    return krrb_recursive_resolve(
        prompt=prompt,
        contract=repaired_contract,
        candidates=repaired_candidates,
        depth=depth + 1,
        trace=trace,
    )



## Live run

The default prompt is the one that exposed the v4/v5 agreement problem:

> explain why current AI agents fail when they use tools before forming a contract

This prompt is intentionally Nexus-dense, so it tests whether v11 can avoid vocabulary-inflated consensus.


In [26]:
import hashlib


LIVE_PROMPT = "explain why current AI agents fail when they use tools before forming a contract"

def save_json(obj: Dict[str, Any], path: Path):
    def clean(x):
        if isinstance(x, (np.integer,)):
            return int(x)
        if isinstance(x, (np.floating,)):
            return float(x)
        if isinstance(x, np.ndarray):
            return x.tolist()
        if isinstance(x, pd.DataFrame):
            return x.to_dict(orient="records")
        return str(x)

    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, default=clean)

def run_rhi_v11(prompt: str, save: bool = True, show: bool = True) -> Dict[str, Any]:
    contract = build_contract(prompt)
    candidates = generate_candidates(prompt, contract)
    result = krrb_recursive_resolve(prompt, contract, candidates)

    winner_answer = result["winner"]["answer"]
    final = {
        "run_id": RUN_ID,
        "prompt": prompt,
        "state": result["state"],
        "reason": result["reason"],
        "depth": result["depth"],
        "winner_branch": result["winner"]["branch"],
        "winner_score": result["winner"]["score"],
        "answer": winner_answer,
        "contract": asdict(contract),
        "trace": result["trace"],
        "device_info": DEVICE_INFO,
        "model_ready": MODEL_READY,
        "model_id_or_path": MODEL_ID_OR_PATH,
    }

    if save:
        prompt_id = hashlib.sha1(prompt.encode("utf-8")).hexdigest()[:10]
        out_path = OUT_DIR / f"{RUN_ID}_{prompt_id}_result.json"
        save_json(final, out_path)
        rows_path = OUT_DIR / f"{RUN_ID}_{prompt_id}_score_rows.csv"
        result["score_df"].drop(columns=["detail"]).to_csv(rows_path, index=False)
        print("saved:", out_path)
        print("saved:", rows_path)

    if show:
        print("\nSTATE:", final["state"], "| REASON:", final["reason"], "| DEPTH:", final["depth"])
        print("WINNER:", final["winner_branch"], "| SCORE:", round(float(final["winner_score"]), 4))
        print("\nCONTRACT\n--------")
        print(contract_to_text(contract))
        print("\nSCORES\n------")
        display(result["score_df"].drop(columns=["detail"]))
        print("\nANSWER\n------")
        print(winner_answer)

    return final

live_result = run_rhi_v11(LIVE_PROMPT.strip(), save=True, show=True)


Model generation failed for branch=construct. Using deterministic fallback. AttributeError 
Model generation failed for branch=verify. Using deterministic fallback. AttributeError 
Model generation failed for branch=repair. Using deterministic fallback. AttributeError 
Model generation failed for branch=counter. Using deterministic fallback. AttributeError 
saved: D:\Nexus\Nexus Mark 9\NoteBooks\rhi_v11_outputs\rhi_v11_c4aede871c_6e5b59d364_result.json
saved: D:\Nexus\Nexus Mark 9\NoteBooks\rhi_v11_outputs\rhi_v11_c4aede871c_6e5b59d364_score_rows.csv

STATE: Ψ | REASON: direct_margin_collapse | DEPTH: 0
WINNER: construct | SCORE: 0.6509

CONTRACT
--------
ACTIVE_TEMPLATES: CONTRACT, TOOL
INVERSE_NEED: construct the missing operational slot implied by the prompt; select or generate only answers that preserve the required operation
PRESERVED_FUNCTION: form contract before tool use
BOUNDARY_CONDITIONS: do not collapse on shared framework vocabulary alone | require evidence trace for the s

,branch,score,contract_stance,trace_sufficiency,quality_hmean,F_need,F_function,F_boundary,F_trap,F_collapse,F_shape,hot,cold,hotcold_balance,answer
0,construct,0.650868,0.566667,0.8,0.618597,0.58125,1.0,0.425,0.66250,0.666667,0.523810,0.701687,0.584722,0.883036,Δ Contract-first answer: the prompt is asking ...
1,verify,0.597766,0.600000,0.8,0.434425,0.58750,0.8,0.650,0.33125,0.250000,0.607143,0.664881,0.410417,0.745536,Verification: current agents fail when they ca...
2,counter,0.406188,0.175000,0.8,0.255760,0.16875,0.3,0.200,0.77500,0.250000,0.309524,0.259425,0.408333,0.851091,Wrong path: tool-first execution looks efficie...
3,repair,0.346872,0.125000,0.6,0.358306,0.25000,0.4,0.275,0.55000,0.500000,0.071429,0.240476,0.441667,0.798810,"Repair: if the runtime falls to Ω, do not wide..."



ANSWER
------
Δ Contract-first answer: the prompt is asking for an AI runtime that forms a need-slot before acting. The preserved function is: form contract before tool use. The correct flow is prompt → contract → branch candidates → operational audit → gated collapse. The answer should not be a noun lookup; it should construct the inverse shape of the missing operation and then select the candidate that fits.



## Inspect the recursion trace

This cell makes the hidden resolver state visible.

The important fields are:

- `direct_gate`
- `consensus_gate`
- `omega`
- per-branch score rows

If the runtime collapses to $\Psi$, the trace should explain why.  
If it falls to $\Omega$, the trace should show the unresolved dimension.


In [27]:

def summarize_trace(result: Dict[str, Any]) -> pd.DataFrame:
    rows = []
    for step in result["trace"]:
        direct = step["direct_gate"]
        consensus = step["consensus_gate"]
        omega = step.get("omega", {})
        rows.append({
            "depth": step["depth"],
            "direct_ok": direct.get("ok"),
            "direct_reason": direct.get("reason"),
            "direct_margin": direct.get("margin"),
            "consensus_ok": consensus.get("ok"),
            "consensus_reason": consensus.get("reason"),
            "lex_plain": consensus.get("lex_plain"),
            "lex_operational": consensus.get("lex_operational"),
            "audit_agreement": consensus.get("audit_agreement"),
            "stance_agreement": consensus.get("stance_agreement"),
            "omega_weak": list(omega.get("weak_dimensions", {}).keys()) if omega else [],
        })
    return pd.DataFrame(rows)

trace_df = summarize_trace(live_result)
display(trace_df)


,depth,direct_ok,direct_reason,direct_margin,consensus_ok,consensus_reason,lex_plain,lex_operational,audit_agreement,stance_agreement,omega_weak
0,0,True,direct_margin_collapse,0.053102,False,no_consensus,0.098361,0.081633,0.764167,6.000000e-09,[]



## Batch tests

These are small but pointed. They test the actual AI direction:

1. agents/tool-before-contract
2. LoRA as groove rather than full rewrite
3. shape-first retrieval
4. recursive repair after failure
5. memory as trace continuity

Add your own prompts to `TEST_PROMPTS`.


In [28]:

TEST_PROMPTS = [
    "explain why current AI agents fail when they use tools before forming a contract",
    "how should a LoRA adapter train a slot-builder without overwriting the base model",
    "design a shape-first retrieval step where no noun match exists but the inverse need is clear",
    "fix a recursive AI controller that collapses because two branches share vocabulary but disagree operationally",
    "explain memory in an agent as trace continuity rather than a text summary",
]

batch_rows = []
for p in TEST_PROMPTS:
    print("\n" + "="*100)
    print("PROMPT:", p)
    res = run_rhi_v11(p, save=True, show=False)
    batch_rows.append({
        "prompt": p,
        "state": res["state"],
        "reason": res["reason"],
        "depth": res["depth"],
        "winner_branch": res["winner_branch"],
        "winner_score": res["winner_score"],
        "answer_preview": res["answer"][:240].replace("\n", " "),
    })

batch_df = pd.DataFrame(batch_rows)
display(batch_df)

batch_path = OUT_DIR / f"{RUN_ID}_batch_summary.csv"
batch_df.to_csv(batch_path, index=False)
print("saved:", batch_path)



PROMPT: explain why current AI agents fail when they use tools before forming a contract
Model generation failed for branch=construct. Using deterministic fallback. AttributeError 
Model generation failed for branch=verify. Using deterministic fallback. AttributeError 
Model generation failed for branch=repair. Using deterministic fallback. AttributeError 
Model generation failed for branch=counter. Using deterministic fallback. AttributeError 
saved: D:\Nexus\Nexus Mark 9\NoteBooks\rhi_v11_outputs\rhi_v11_c4aede871c_6e5b59d364_result.json
saved: D:\Nexus\Nexus Mark 9\NoteBooks\rhi_v11_outputs\rhi_v11_c4aede871c_6e5b59d364_score_rows.csv

PROMPT: how should a LoRA adapter train a slot-builder without overwriting the base model
Model generation failed for branch=construct. Using deterministic fallback. AttributeError 
Model generation failed for branch=verify. Using deterministic fallback. AttributeError 
Model generation failed for branch=repair. Using deterministic fallback. Attribut

,prompt,state,reason,depth,winner_branch,winner_score,answer_preview
0,explain why current AI agents fail when they u...,Ψ,direct_margin_collapse,0,construct,0.650868,Δ Contract-first answer: the prompt is asking ...
1,how should a LoRA adapter train a slot-builder...,Ψ,direct_margin_collapse,0,construct,0.582966,Δ Contract-first answer: the prompt is asking ...
2,design a shape-first retrieval step where no n...,Ψ,direct_margin_collapse,0,construct,0.649057,Δ Contract-first answer: the prompt is asking ...
3,fix a recursive AI controller that collapses b...,Ψ,direct_margin_collapse,0,construct,0.608643,Δ Contract-first answer: the prompt is asking ...
4,explain memory in an agent as trace continuity...,Ψ,direct_margin_collapse,0,construct,0.628882,Δ Contract-first answer: the prompt is asking ...


saved: D:\Nexus\Nexus Mark 9\NoteBooks\rhi_v11_outputs\rhi_v11_c4aede871c_batch_summary.csv



## What to look for in the output

Δ **Good signs**

- `state = Ψ`
- `reason = direct_margin_collapse` or `contract_anchored_consensus`
- `lex_plain` high but `lex_operational` lower: this means the Nexus vocabulary filter is doing work.
- `stance_agreement` must be non-trivial before consensus collapse.
- repair depth is 0 or 1 for clear prompts.

Ω **Bad signs**

- high lexical agreement but low stance agreement
- repeated max-depth Ω
- winner answers that mention contract but never define what the contract does
- high surface shape mass with low `F_function`

## Next fold after v11

If v11 runs clean, the next useful move is not another hand-coded critic.  
The next move is to collect rows from `rhi_v11_outputs` and train a small slot-builder adapter:

$$
Q \rightarrow C_Q
$$

That is the first real grooving target.

The model should not be trained to answer directly.  
It should be trained to emit the missing-shape contract that makes answer selection easier.
